# 3.4 Applications of Determinants

**Course:** Elementary Linear Algebra (Larson, 8e)  
**Chapter 3:** Determinants

---

### What you will learn

- How to use **Cramer's Rule** to solve $n \times n$ linear systems via determinants
- How to compute the **adjoint** (classical adjugate) of a square matrix
- How to find the **inverse** using $A^{-1} = \frac{1}{\det A}\operatorname{adj}(A)$
- How to compute the **area of a triangle** from vertex coordinates
- How to test whether points are **collinear** using a determinant condition
- How to find the **equation of a line** through two points via a determinant formula
- How to compute the **volume of a tetrahedron** from edge vectors

---

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))

In [ ]:
from linalg_core.matrix import Matrix
from linalg_core.ops import multiply, transpose, inverse, scalar_mul
from linalg_core.determinant import (
    det_cofactor,
    det_elimination,
    cofactor_matrix,
    adjoint,
    inverse_adjoint,
    cramers_rule,
)
from linalg_core.elimination import solve
from linalg_core import EPSILON

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

---

## 2. Motivation --- Surveyor's Problem

Surveyors have marked three points in the plane:

$$(1, 2), \quad (4, 6), \quad (7, 10)$$

Two immediate questions arise:

1. **Are these points collinear?** If they all lie on the same line, the
   "triangle" they form has zero area --- not a useful plot boundary.
2. **Adding a fourth point** $(2, 5)$ that is *not* on that line --- what is
   the area of the triangle formed by $(1,2)$, $(2,5)$, and $(4,6)$?

These are geometry questions, but they have clean answers expressed as
**determinants**. This notebook develops the tools to answer them, along
with several other applications of the determinant from Chapter 3.

In [ ]:
surveyor_pts = [(1, 2), (4, 6), (7, 10)]
extra_pt = (2, 5)

print("Surveyor points:", surveyor_pts)
print("Extra point:    ", extra_pt)

---

## 3. Build --- Core Concepts

We develop seven applications of determinants:

1. Cramer's Rule
2. The adjoint matrix
3. Inverse via the adjoint
4. Area of a triangle
5. Collinearity test
6. Equation of a line
7. Volume of a tetrahedron

### 3.1 Cramer's Rule

**Theorem (Cramer's Rule).** Let $A$ be an $n \times n$ invertible matrix
and let $\mathbf{b} \in \mathbb{R}^n$. Then the unique solution of
$A\mathbf{x} = \mathbf{b}$ has components

$$
x_i = \frac{\det(A_i)}{\det(A)}, \qquad i = 1, 2, \ldots, n
$$

where $A_i$ is the matrix obtained by **replacing column $i$** of $A$ with
$\mathbf{b}$.

**Why it works (first-principles sketch).** Expanding $\det(A_i)$ along
column $i$ is equivalent to taking the dot product of $\mathbf{b}$ with the
cofactors of column $i$. Because $A^{-1} = \frac{1}{\det A}\operatorname{adj}(A)$,
row $i$ of $A^{-1}$ is exactly those cofactors divided by $\det A$.

**Computational cost.** Cramer's Rule requires computing $n + 1$ determinants
of $n \times n$ matrices, giving $O(n \cdot n!) $ by cofactor expansion or
$O(n^4)$ using elimination-based determinants. For large systems, Gaussian
elimination ($O(n^3)$) is far more efficient. Cramer's Rule is primarily a
theoretical tool and useful for small systems.

In [ ]:
A = Matrix.from_lists([
    [2, -1,  5],
    [1,  1, -3],
    [2,  4,  1],
])
b = [10, -2, 1]

print("Coefficient matrix A:")
print(A)
print("\nRight-hand side b:", b)
print("\ndet(A) =", det_elimination(A))

x_cramer = cramers_rule(A, b)
print("\nCramer's Rule solution:")
for i, xi in enumerate(x_cramer):
    print(f"  x{i+1} = {xi:.6f}")

print("\nStep-by-step:")
det_A = det_elimination(A)
for i in range(A.rows):
    Ai = A.copy()
    for r in range(A.rows):
        Ai[r, i] = b[r]
    det_Ai = det_elimination(Ai)
    print(f"  det(A{i+1}) = {det_Ai:.4f}  =>  x{i+1} = {det_Ai:.4f} / {det_A:.4f} = {det_Ai/det_A:.6f}")

### 3.2 The Adjoint Matrix

The **adjoint** (or **classical adjugate**) of an $n \times n$ matrix $A$
is the transpose of the matrix of cofactors:

$$
\operatorname{adj}(A) = C^T
$$

where $C_{ij} = (-1)^{i+j} M_{ij}$ and $M_{ij}$ is the $(i,j)$ minor.

**Key identity:**

$$
A \cdot \operatorname{adj}(A) = \operatorname{adj}(A) \cdot A = (\det A)\, I
$$

This identity is the bridge that connects determinants to the inverse.

In [ ]:
print("Matrix A:")
print(A)

C = cofactor_matrix(A)
print("\nCofactor matrix C:")
print(C)

adj_A = adjoint(A)
print("\nAdjoint adj(A) = C^T:")
print(adj_A)

product = multiply(A, adj_A)
print("\nA * adj(A):")
print(product)

det_val = det_cofactor(A)
print(f"\ndet(A) = {det_val:.4f}")
print(f"det(A) * I should match A * adj(A):")
det_I = scalar_mul(Matrix.identity(3), det_val)
print(det_I)
print(f"\nMatrices equal: {product == det_I}")

### 3.3 Inverse via the Adjoint

From the identity $A \cdot \operatorname{adj}(A) = (\det A)\, I$, dividing
both sides by $\det A$ (assuming $\det A \ne 0$) gives:

$$
A^{-1} = \frac{1}{\det A}\, \operatorname{adj}(A)
$$

This is the **adjoint formula** for the inverse. It is elegant and exact
for symbolic work, but computationally expensive --- $O(n \cdot n!)$ via
cofactor expansion versus $O(n^3)$ for Gauss-Jordan.

We verify that `inverse_adjoint(A)` agrees with `inverse(A)` (Gauss-Jordan).

In [ ]:
A_inv_adj = inverse_adjoint(A)
print("A^{-1} via adjoint formula:")
print(A_inv_adj)

A_inv_gj = inverse(A)
print("\nA^{-1} via Gauss-Jordan:")
print(A_inv_gj)

print(f"\nMatrices equal: {A_inv_adj == A_inv_gj}")

should_be_I = multiply(A, A_inv_adj)
print("\nA * A^{-1} (verification):")
print(should_be_I)
print(f"Equals identity: {should_be_I == Matrix.identity(3)}")

### 3.4 Area of a Triangle

Given three vertices $(x_1, y_1)$, $(x_2, y_2)$, $(x_3, y_3)$, the
**signed area** of the triangle they form is:

$$
\text{Area} = \frac{1}{2} \left| \det \begin{bmatrix}
x_1 & y_1 & 1 \\
x_2 & y_2 & 1 \\
x_3 & y_3 & 1
\end{bmatrix} \right|
$$

**Why it works.** The third column of ones converts vertex coordinates into
an affine framework. Expanding the determinant yields the standard
**shoelace formula**:

$$
\text{Area} = \tfrac{1}{2} |x_1(y_2 - y_3) + x_2(y_3 - y_1) + x_3(y_1 - y_2)|
$$

In [ ]:
def triangle_area(p1, p2, p3):
    """Area of the triangle with vertices p1, p2, p3.

    Each vertex is a tuple (x, y). Uses the determinant formula.
    """
    M = Matrix.from_lists([
        [p1[0], p1[1], 1],
        [p2[0], p2[1], 1],
        [p3[0], p3[1], 1],
    ])
    return abs(det_cofactor(M)) / 2.0


def shoelace_area(p1, p2, p3):
    """Area via the shoelace formula (for verification)."""
    x1, y1 = p1
    x2, y2 = p2
    x3, y3 = p3
    return abs(x1*(y2 - y3) + x2*(y3 - y1) + x3*(y1 - y2)) / 2.0


p1, p2, p3 = (1, 2), (2, 5), (4, 6)

area_det = triangle_area(p1, p2, p3)
area_shoe = shoelace_area(p1, p2, p3)

print(f"Triangle vertices: {p1}, {p2}, {p3}")
print(f"\nArea (determinant formula): {area_det:.6f}")
print(f"Area (shoelace formula):    {area_shoe:.6f}")
print(f"\nAreas match: {abs(area_det - area_shoe) < EPSILON}")

### 3.5 Collinearity Test

Three points are **collinear** (lie on the same line) if and only if
the triangle they form has **zero area**. Equivalently:

$$
\det \begin{bmatrix}
x_1 & y_1 & 1 \\
x_2 & y_2 & 1 \\
x_3 & y_3 & 1
\end{bmatrix} = 0
\quad \Longleftrightarrow \quad
\text{points are collinear}
$$

Let us test the surveyor's three points $(1,2)$, $(4,6)$, $(7,10)$.

In [ ]:
def are_collinear(p1, p2, p3):
    """Test if three 2D points are collinear using the determinant test."""
    M = Matrix.from_lists([
        [p1[0], p1[1], 1],
        [p2[0], p2[1], 1],
        [p3[0], p3[1], 1],
    ])
    return abs(det_cofactor(M)) < EPSILON


s1, s2, s3 = surveyor_pts
print(f"Surveyor points: {s1}, {s2}, {s3}")

M_surv = Matrix.from_lists([
    [s1[0], s1[1], 1],
    [s2[0], s2[1], 1],
    [s3[0], s3[1], 1],
])
det_surv = det_cofactor(M_surv)
print(f"\nDeterminant = {det_surv:.6f}")
print(f"Collinear: {are_collinear(s1, s2, s3)}")

area_surv = triangle_area(s1, s2, s3)
print(f"Triangle area = {area_surv:.6f}  (confirms area ~ 0)")

print("\n--- Now test with the extra point ---")
print(f"Points: {s1}, {extra_pt}, {s2}")
print(f"Collinear: {are_collinear(s1, extra_pt, s2)}")
print(f"Triangle area = {triangle_area(s1, extra_pt, s2):.6f}")

### 3.6 Equation of a Line Through Two Points

Given two points $(x_1, y_1)$ and $(x_2, y_2)$, the equation of the
**line passing through them** can be written as:

$$
\det \begin{bmatrix}
x  & y  & 1 \\
x_1 & y_1 & 1 \\
x_2 & y_2 & 1
\end{bmatrix} = 0
$$

Expanding the determinant along the first row gives the **implicit form**
$ax + by + c = 0$, where $a$, $b$, $c$ are cofactors of the first row.

In [ ]:
def line_equation(p1, p2):
    """Return (a, b, c) for the line ax + by + c = 0 through p1 and p2.

    Derived from the determinant expansion of [[x,y,1],[x1,y1,1],[x2,y2,1]] = 0.
    """
    x1, y1 = p1
    x2, y2 = p2
    a = (y1 - y2)
    b = (x2 - x1)
    c = (x1 * y2 - x2 * y1)
    return a, b, c


p_line1 = (1, 2)
p_line2 = (4, 6)

a, b, c = line_equation(p_line1, p_line2)
print(f"Line through {p_line1} and {p_line2}:")
print(f"  {a}x + {b}y + {c} = 0")

if abs(b) > EPSILON:
    print(f"  Slope-intercept form: y = {-a/b:.4f}x + {-c/b:.4f}")

print("\nVerification --- substitute both points:")
for p in [p_line1, p_line2]:
    val = a * p[0] + b * p[1] + c
    print(f"  {a}*{p[0]} + {b}*{p[1]} + {c} = {val:.6f}")

print("\nVerification --- third collinear surveyor point (7, 10):")
p_test = (7, 10)
val = a * p_test[0] + b * p_test[1] + c
print(f"  {a}*{p_test[0]} + {b}*{p_test[1]} + {c} = {val:.6f}  (should be 0 since collinear)")

print("\nNon-collinear point (2, 5):")
val_nc = a * extra_pt[0] + b * extra_pt[1] + c
print(f"  {a}*{extra_pt[0]} + {b}*{extra_pt[1]} + {c} = {val_nc:.6f}  (non-zero => not on line)")

### 3.7 Volume of a Tetrahedron

Given four vertices $P_0, P_1, P_2, P_3$ in $\mathbb{R}^3$, form the
three edge vectors from $P_0$:

$$
\mathbf{u} = P_1 - P_0, \quad
\mathbf{v} = P_2 - P_0, \quad
\mathbf{w} = P_3 - P_0
$$

The volume of the tetrahedron is:

$$
V = \frac{1}{6} \left| \det \begin{bmatrix}
u_1 & u_2 & u_3 \\
v_1 & v_2 & v_3 \\
w_1 & w_2 & w_3
\end{bmatrix} \right|
$$

**Why 1/6?** A parallelepiped has volume $|\det[\mathbf{u}, \mathbf{v}, \mathbf{w}]|$.
A tetrahedron is $\frac{1}{6}$ of that parallelepiped (just as a triangle is
$\frac{1}{2}$ of a parallelogram).

In [ ]:
def tetrahedron_volume(p0, p1, p2, p3):
    """Volume of the tetrahedron with vertices p0, p1, p2, p3 in R^3.

    Each vertex is a tuple/list of 3 coordinates.
    """
    u = [p1[i] - p0[i] for i in range(3)]
    v = [p2[i] - p0[i] for i in range(3)]
    w = [p3[i] - p0[i] for i in range(3)]
    M = Matrix.from_lists([u, v, w])
    return abs(det_cofactor(M)) / 6.0


P0 = (0, 0, 0)
P1 = (1, 0, 0)
P2 = (0, 1, 0)
P3 = (0, 0, 1)

vol = tetrahedron_volume(P0, P1, P2, P3)
print(f"Tetrahedron vertices:")
print(f"  P0 = {P0}")
print(f"  P1 = {P1}")
print(f"  P2 = {P2}")
print(f"  P3 = {P3}")
print(f"\nEdge vectors from P0:")
u = [P1[i] - P0[i] for i in range(3)]
v = [P2[i] - P0[i] for i in range(3)]
w = [P3[i] - P0[i] for i in range(3)]
print(f"  u = P1 - P0 = {u}")
print(f"  v = P2 - P0 = {v}")
print(f"  w = P3 - P0 = {w}")

M_tet = Matrix.from_lists([u, v, w])
print(f"\nEdge matrix:")
print(M_tet)
print(f"\ndet = {det_cofactor(M_tet):.6f}")
print(f"Volume = |det| / 6 = {vol:.6f}")
print(f"\nExpected: 1/6 = {1/6:.6f}")

print("\n--- Non-trivial tetrahedron ---")
Q0 = (1, 1, 1)
Q1 = (4, 2, 1)
Q2 = (2, 5, 1)
Q3 = (1, 2, 4)
vol2 = tetrahedron_volume(Q0, Q1, Q2, Q3)
print(f"Vertices: {Q0}, {Q1}, {Q2}, {Q3}")
print(f"Volume = {vol2:.6f}")

np_vol = abs(np.linalg.det(np.array([
    [Q1[i] - Q0[i] for i in range(3)],
    [Q2[i] - Q0[i] for i in range(3)],
    [Q3[i] - Q0[i] for i in range(3)],
]))) / 6.0
print(f"NumPy verification: {np_vol:.6f}")

---

## 4. Verify

We run three independent cross-checks to confirm our implementations.

### 4.1 Cramer's Rule vs. RREF vs. NumPy

In [ ]:
print("=== Cross-verification: Cramer's Rule vs RREF vs NumPy ===")
print()

x_cramer = cramers_rule(A, b)

aug_data = [A.get_row(i) + [b[i]] for i in range(A.rows)]
aug = Matrix.from_lists(aug_data)
sol_type, x_rref = solve(aug)

A_np = np.array(A.to_lists())
b_np = np.array(b)
x_np = np.linalg.solve(A_np, b_np)

print(f"{'Method':<20} {'x1':>10} {'x2':>10} {'x3':>10}")
print("-" * 52)
print(f"{'Cramers Rule':<20} {x_cramer[0]:>10.6f} {x_cramer[1]:>10.6f} {x_cramer[2]:>10.6f}")
print(f"{'RREF':<20} {x_rref[0]:>10.6f} {x_rref[1]:>10.6f} {x_rref[2]:>10.6f}")
print(f"{'NumPy':<20} {x_np[0]:>10.6f} {x_np[1]:>10.6f} {x_np[2]:>10.6f}")

cramer_rref_match = all(abs(a - b) < EPSILON for a, b in zip(x_cramer, x_rref))
cramer_np_match = all(abs(a - b) < 1e-6 for a, b in zip(x_cramer, x_np))
print(f"\nCramer == RREF:  {cramer_rref_match}")
print(f"Cramer == NumPy: {cramer_np_match}")

### 4.2 Inverse (Adjoint) vs. Gauss-Jordan

In [ ]:
print("=== Cross-verification: Inverse (Adjoint vs Gauss-Jordan) ===")
print()

A_inv_adj = inverse_adjoint(A)
A_inv_gj = inverse(A)

A_np_inv = np.linalg.inv(A_np)

print("Adjoint inverse:")
print(A_inv_adj)
print("\nGauss-Jordan inverse:")
print(A_inv_gj)

print(f"\nAdjoint == Gauss-Jordan: {A_inv_adj == A_inv_gj}")

max_diff = max(
    abs(A_inv_adj[i, j] - A_np_inv[i][j])
    for i in range(3) for j in range(3)
)
print(f"Max |adjoint - NumPy|:  {max_diff:.2e}")

### 4.3 Triangle Area: Determinant vs. Shoelace, and Collinear => Area ~ 0

In [ ]:
print("=== Cross-verification: Triangle Area ===")
print()

test_triangles = [
    ((0, 0), (4, 0), (0, 3), "Right triangle (3-4-5)"),
    ((1, 2), (2, 5), (4, 6), "Surveyor triangle"),
    ((1, 1), (3, 1), (2, 4), "Isosceles-ish"),
]

print(f"{'Triangle':<30} {'Det':>10} {'Shoelace':>10} {'Match':>7}")
print("-" * 60)
for pa, pb, pc, label in test_triangles:
    a_det = triangle_area(pa, pb, pc)
    a_shoe = shoelace_area(pa, pb, pc)
    match = abs(a_det - a_shoe) < EPSILON
    print(f"{label:<30} {a_det:>10.4f} {a_shoe:>10.4f} {str(match):>7}")

print("\n--- Collinear points => area ~ 0 ---")
collinear_area = triangle_area((1, 2), (4, 6), (7, 10))
print(f"Area of 'triangle' (1,2)-(4,6)-(7,10) = {collinear_area:.10f}")
print(f"Area approximately zero: {collinear_area < EPSILON}")

---

## 5. Visualize

### 5.1 Triangle with Shaded Area

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 6))

tri_pts = [(1, 2), (2, 5), (4, 6)]
area = triangle_area(*tri_pts)

xs = [p[0] for p in tri_pts] + [tri_pts[0][0]]
ys = [p[1] for p in tri_pts] + [tri_pts[0][1]]

ax.fill([p[0] for p in tri_pts], [p[1] for p in tri_pts],
        alpha=0.25, color='steelblue', label=f'Area = {area:.2f}')
ax.plot(xs, ys, 'o-', color='steelblue', markersize=8, linewidth=2)

for p in tri_pts:
    ax.annotate(f'({p[0]}, {p[1]})', xy=p, fontsize=11,
                textcoords='offset points', xytext=(8, 8))

ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('Triangle Area via Determinant', fontsize=14)
ax.legend(fontsize=12)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5.2 Collinear Points on a Line

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5))

col_pts = [(1, 2), (4, 6), (7, 10)]
a_line, b_line, c_line = line_equation(col_pts[0], col_pts[1])

x_range = np.linspace(-1, 9, 200)
if abs(b_line) > EPSILON:
    y_range = (-a_line * x_range - c_line) / b_line
    ax.plot(x_range, y_range, '--', color='gray', linewidth=1.5,
            label=f'{a_line}x + {b_line}y + {c_line} = 0')

for p in col_pts:
    ax.plot(p[0], p[1], 's', color='crimson', markersize=10, zorder=5)
    ax.annotate(f'({p[0]}, {p[1]})', xy=p, fontsize=11,
                textcoords='offset points', xytext=(8, -14))

ep = extra_pt
ax.plot(ep[0], ep[1], 'D', color='forestgreen', markersize=10, zorder=5)
ax.annotate(f'({ep[0]}, {ep[1]}) --- not collinear', xy=ep, fontsize=11,
            textcoords='offset points', xytext=(10, 8), color='forestgreen')

ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('Collinearity Test: det = 0 iff on the same line', fontsize=14)
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5.3 Tetrahedron in 3D

In [ ]:
fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(111, projection='3d')

verts = np.array([
    [0, 0, 0],
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 1],
])

faces = [
    [verts[0], verts[1], verts[2]],
    [verts[0], verts[1], verts[3]],
    [verts[0], verts[2], verts[3]],
    [verts[1], verts[2], verts[3]],
]

poly = Poly3DCollection(faces, alpha=0.2, facecolor='steelblue',
                        edgecolor='navy', linewidth=1.5)
ax.add_collection3d(poly)

labels = ['$P_0$', '$P_1$', '$P_2$', '$P_3$']
for i, (v, lab) in enumerate(zip(verts, labels)):
    ax.scatter(*v, color='crimson', s=60, zorder=5)
    ax.text(v[0] + 0.05, v[1] + 0.05, v[2] + 0.05, lab, fontsize=13)

vol_unit = tetrahedron_volume(tuple(verts[0]), tuple(verts[1]),
                              tuple(verts[2]), tuple(verts[3]))

ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_zlabel('z', fontsize=12)
ax.set_title(f'Unit Tetrahedron  (V = {vol_unit:.4f})', fontsize=14)
ax.set_xlim(-0.2, 1.3)
ax.set_ylim(-0.2, 1.3)
ax.set_zlim(-0.2, 1.3)
plt.tight_layout()
plt.show()

---

## 6. Exercises

### Exercise 1 (Easy) --- Cramer's Rule for a 2x2 System

Use `cramers_rule` to solve the $2 \times 2$ system:

$$
\begin{cases}
3x + 2y = 7 \\
x - y = 1
\end{cases}
$$

Then verify your answer by substituting back into both equations.

**Hint:** Build `A = Matrix.from_lists([[3, 2], [1, -1]])` and `b = [7, 1]`.

In [ ]:
# YOUR CODE HERE


### Exercise 2 (Medium) --- Collinearity and Area

You are given four points:

$$A = (0, 0), \quad B = (6, 2), \quad C = (3, 1), \quad D = (4, 5)$$

1. Test whether $A$, $B$, $C$ are collinear.
2. Compute the area of triangle $A B D$.
3. Use the `line_equation` function to find the equation of line $AB$,
   and confirm that $C$ satisfies it but $D$ does not.

**Hint:** For part 1, use `are_collinear`. For part 2, use `triangle_area`.

In [ ]:
# YOUR CODE HERE


### Exercise 3 (Challenge) --- Equation of a Plane

Extend the 2D line-equation technique to 3D. Three non-collinear points
$P_1 = (x_1, y_1, z_1)$, $P_2 = (x_2, y_2, z_2)$, $P_3 = (x_3, y_3, z_3)$
determine a plane. The equation of that plane can be written as:

$$
\det \begin{bmatrix}
x - x_1 & y - y_1 & z - z_1 \\
x_2 - x_1 & y_2 - y_1 & z_2 - z_1 \\
x_3 - x_1 & y_3 - y_1 & z_3 - z_1
\end{bmatrix} = 0
$$

Write a function `equation_of_plane(p1, p2, p3)` that returns `(a, b, c, d)`
for the plane $ax + by + cz + d = 0$.

**Test it** with $P_1 = (1, 0, 0)$, $P_2 = (0, 1, 0)$, $P_3 = (0, 0, 1)$.
The expected plane is $x + y + z = 1$ (i.e., $a = b = c = 1$, $d = -1$, or
any scalar multiple).

**Hint:** Expand the $3 \times 3$ determinant symbolically. The coefficients
$a$, $b$, $c$ are the cofactors of the first row, and $d = -(a x_1 + b y_1 + c z_1)$.

In [ ]:
# YOUR CODE HERE


---

## Summary

| Concept | Key Formula / Takeaway |
|---|---|
| **Cramer's Rule** | $x_i = \det(A_i) / \det(A)$; elegant but $O(n^4)$ --- use for small systems or theory |
| **Adjoint matrix** | $\operatorname{adj}(A) = C^T$; satisfies $A \cdot \operatorname{adj}(A) = (\det A)\, I$ |
| **Inverse via adjoint** | $A^{-1} = \frac{1}{\det A} \operatorname{adj}(A)$; exact but expensive |
| **Triangle area** | $\frac{1}{2}|\det[\text{vertices augmented with 1s}]|$; equivalent to shoelace |
| **Collinearity** | $\det = 0 \iff$ points collinear $\iff$ area $= 0$ |
| **Line equation** | $\det[[x,y,1],[x_1,y_1,1],[x_2,y_2,1]] = 0$ gives implicit form |
| **Tetrahedron volume** | $V = \frac{1}{6}|\det[\text{edge vectors}]|$; extends triangle area to 3D |